# 가중치 초기화 전략 - Xavier, He, GPT-2 초기화

- Tutorial ID: `adv-6-1`
- Tutorial: 가중치 초기화 전략
- Section ID: `adv-6-1-1`
- Section: Xavier, He, GPT-2 초기화


In [ ]:
# ============================================================
# 코드 읽는 법 — Xavier, He, GPT-2 초기화
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("가중치 초기화 전략")
print("=" * 60)

d = 512
n_layers = 50
np.random.seed(42)

In [ ]:
# 1. 나쁜 초기화

In [ ]:
print("\n1. 스케일링 없는 초기화 (폭발!)")
x = np.random.randn(d)
for i in range(n_layers):
    W = np.random.randn(d, d)
        x = W @ x
    if i < 3 or i >= n_layers - 2:
        print(f"  레이어 {i+1}: 분산 = {np.var(x):.2e}")

In [ ]:
# 2. Xavier 초기화

In [ ]:
print("\n2. Xavier 초기화 (분산 유지)")
x = np.random.randn(d)
for i in range(n_layers):
    std = np.sqrt(2.0 / (d + d))
    W = np.random.randn(d, d) * std
        x = W @ x
    if i < 3 or i >= n_layers - 2:
        print(f"  레이어 {i+1}: 분산 = {np.var(x):.4f}")

In [ ]:
# 3. GPT-2 스타일

In [ ]:
print("\n3. GPT-2 스타일 초기화")
n_layers_gpt = 12
d_model = 768

std_normal = 0.02
std_residual = 0.02 / np.sqrt(2 * n_layers_gpt)

print(f"  일반 가중치 std: {std_normal}")
print(f"  잔차 출력 std: {std_residual:.6f}")
print(f"  → 잔차 누적 분산 ≈ n_layers를 1/√(2L) 스케일링으로 보상")

# 검증
x = np.random.randn(4, 32, d_model)
print(f"\n전방 패스 분산 추적:")
print(f"  입력: {np.var(x):.4f}")
for l in range(6):
    W = np.random.randn(d_model, d_model) * std_residual
        attn_out = x @ W
        x = x + attn_out
    print(f"  레이어 {l+1}: {np.var(x):.4f}")
print(f"  → 분산이 안정적으로 유지!")